# Data Exploration for Deepfake Detection

This notebook explores the deepfake datasets used in this project:
- FaceForensics++
- Celeb-DF
- DFDC (Deepfake Detection Challenge)

We'll analyze:
1. Dataset statistics and distribution
2. Sample visualization (real vs fake)
3. Data quality analysis
4. Class balance checking

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import yaml

from data import get_dataset, get_augmentation_pipeline
from utils.visualization import plot_confusion_matrix

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Configuration

In [ ]:
# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"Train dataset: {config['data']['train_dataset']}")
print(f"Test datasets: {config['data']['test_datasets']}")
print(f"Image size: {config['data']['image_size']}")
print(f"Batch size: {config['data']['batch_size']}")

## 2. Dataset Statistics

Let's check the number of samples in each dataset and split.

In [ ]:
def count_dataset_samples(dataset_name, data_root, splits=['train', 'val', 'test']):
    """Count samples in each split of a dataset."""
    stats = {}
    
    for split in splits:
        try:
            dataset = get_dataset(
                dataset_name=dataset_name,
                root_dir=str(Path(data_root) / dataset_name.replace('+', '')),
                split=split,
                transform=None,
                image_size=224
            )
            
            # Count real and fake samples
            labels = dataset.labels
            num_real = labels.count(0)
            num_fake = labels.count(1)
            
            stats[split] = {
                'total': len(dataset),
                'real': num_real,
                'fake': num_fake,
                'balance': num_fake / (num_real + num_fake) if (num_real + num_fake) > 0 else 0
            }
        except Exception as e:
            print(f"Error loading {dataset_name} {split}: {e}")
            stats[split] = None
    
    return stats

# Count samples for each dataset
data_root = config['data']['data_root']
datasets_to_analyze = config['data']['test_datasets']

all_stats = {}
for dataset_name in datasets_to_analyze:
    print(f"\nAnalyzing {dataset_name}...")
    stats = count_dataset_samples(dataset_name, data_root)
    all_stats[dataset_name] = stats
    
    # Print statistics
    for split, split_stats in stats.items():
        if split_stats:
            print(f"  {split.capitalize()}: {split_stats['total']} samples")
            print(f"    Real: {split_stats['real']} ({split_stats['real']/split_stats['total']*100:.1f}%)")
            print(f"    Fake: {split_stats['fake']} ({split_stats['fake']/split_stats['total']*100:.1f}%)")

## 3. Visualize Dataset Distribution

In [ ]:
# Prepare data for visualization
fig, axes = plt.subplots(1, len(datasets_to_analyze), figsize=(15, 5))

if len(datasets_to_analyze) == 1:
    axes = [axes]

for idx, dataset_name in enumerate(datasets_to_analyze):
    stats = all_stats[dataset_name]
    
    # Aggregate across splits
    total_real = sum(s['real'] for s in stats.values() if s)
    total_fake = sum(s['fake'] for s in stats.values() if s)
    
    # Create pie chart
    axes[idx].pie(
        [total_real, total_fake],
        labels=['Real', 'Fake'],
        autopct='%1.1f%%',
        colors=['lightgreen', 'lightcoral'],
        startangle=90
    )
    axes[idx].set_title(f'{dataset_name}\nTotal: {total_real + total_fake}', fontweight='bold')

plt.suptitle('Dataset Class Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Sample Visualization

Let's visualize some real and fake samples from each dataset.

In [ ]:
def visualize_samples(dataset, num_samples=8, title='Samples'):
    """Visualize random samples from dataset."""
    if len(dataset) == 0:
        print(f"No samples in dataset for {title}")
        return
    
    # Get random indices
    indices = np.random.choice(len(dataset), min(num_samples, len(dataset)), replace=False)
    
    rows = 2
    cols = num_samples // rows
    fig, axes = plt.subplots(rows, cols, figsize=(16, 6))
    axes = axes.flatten()
    
    for i, idx in enumerate(indices):
        img_path = dataset.samples[idx]
        label = dataset.labels[idx]
        
        # Load and display image
        img = Image.open(img_path).convert('RGB')
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(
            f"{'Real' if label == 0 else 'Fake'}",
            color='green' if label == 0 else 'red',
            fontweight='bold'
        )
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualize samples from each dataset
for dataset_name in datasets_to_analyze:
    try:
        dataset = get_dataset(
            dataset_name=dataset_name,
            root_dir=str(Path(data_root) / dataset_name.replace('+', '')),
            split='train',
            transform=None,
            image_size=224
        )
        
        visualize_samples(dataset, num_samples=8, title=f'{dataset_name} - Sample Images')
    except Exception as e:
        print(f"Error visualizing {dataset_name}: {e}")

## 5. Augmentation Preview

Let's visualize how different augmentation techniques affect images.

In [ ]:
# Load a sample image
dataset_name = config['data']['train_dataset']
dataset = get_dataset(
    dataset_name=dataset_name,
    root_dir=str(Path(data_root) / dataset_name.replace('+', '')),
    split='train',
    transform=None,
    image_size=224
)

if len(dataset) > 0:
    # Get a sample image
    sample_img_path = dataset.samples[0]
    sample_img = np.array(Image.open(sample_img_path).convert('RGB'))
    
    # Get augmentation pipelines
    basic_aug = get_augmentation_pipeline('basic', 224, config['augmentation'], is_training=True)
    robust_aug = get_augmentation_pipeline('robust', 224, config['augmentation'], is_training=True)
    
    # Apply augmentations multiple times
    fig, axes = plt.subplots(3, 5, figsize=(15, 9))
    
    # Original
    axes[0, 0].imshow(sample_img)
    axes[0, 0].set_title('Original', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Basic augmentation
    for i in range(1, 5):
        aug_img = basic_aug(image=sample_img.copy())['image']
        # Convert tensor to numpy and denormalize
        aug_img = aug_img.permute(1, 2, 0).numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        aug_img = aug_img * std + mean
        aug_img = np.clip(aug_img, 0, 1)
        
        axes[1, i].imshow(aug_img)
        axes[1, i].set_title(f'Basic Aug {i}', fontweight='bold')
        axes[1, i].axis('off')
    
    # Robust augmentation
    for i in range(5):
        aug_img = robust_aug(image=sample_img.copy())['image']
        # Convert tensor to numpy and denormalize
        aug_img = aug_img.permute(1, 2, 0).numpy()
        aug_img = aug_img * std + mean
        aug_img = np.clip(aug_img, 0, 1)
        
        axes[2, i].imshow(aug_img)
        axes[2, i].set_title(f'Robust Aug {i}', fontweight='bold')
        axes[2, i].axis('off')
    
    # Remove unused subplots
    for i in range(1, 5):
        axes[0, i].axis('off')
    axes[1, 0].axis('off')
    
    plt.suptitle('Augmentation Preview: Original vs Basic vs Robust', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("No samples available for augmentation preview")

## 6. Summary

Key observations from data exploration:
1. Dataset sizes and class balance
2. Visual differences between datasets
3. Impact of augmentation strategies

In [ ]:
print("Data Exploration Complete!")
print("\nKey Findings:")
print("- All datasets contain both real and fake samples")
print("- Augmentation strategies vary in intensity")
print("- Robust augmentation applies multiple transformations for better generalization")